In [2]:
import pandas as pd
import os

# --- Paths ---
elpac_dir = "../1095 Data/ELPAC/"

# --- Load all ELPAC files ---
elpac_files = {
    "Initial 2022":   "InitialElpac_22.csv",
    "Initial 2023":   "InitialElpac_23.csv",
    "Initial 2024":   "InitialElpac_24.csv",
    "Summative 2022": "SummativeElpac_22.csv",
    "Summative 2023": "SummativeElpac_23.csv",
    "Summative 2024": "SummativeElpac_24.csv",
}

dfs = {}
for label, filename in elpac_files.items():
    df = pd.read_csv(os.path.join(elpac_dir, filename))
    df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    df["OverallScaleScore"] = pd.to_numeric(df["OverallScaleScore"], errors="coerce")
    df["TestStartDate"] = pd.to_datetime(df["TestStartDate"], errors="coerce")
    dfs[label] = df

# --- Duplicate analysis ---
print("=" * 65)
print(f"{'File':<20} {'Total Rows':>10} {'Unique SSIDs':>13} {'Duplicates':>11} {'Dupe %':>8}")
print("=" * 65)
for label, df in dfs.items():
    total = len(df)
    unique = df["SSID"].nunique()
    dupes = total - unique
    pct = dupes / total * 100
    print(f"{label:<20} {total:>10,} {unique:>13,} {dupes:>11,} {pct:>7.1f}%")

print()

# --- Breakdown: how many SSIDs have 2, 3, 4+ records ---
print("=" * 65)
print("Duplicate breakdown (number of records per SSID):")
print("=" * 65)
for label, df in dfs.items():
    counts = df.groupby("SSID").size().value_counts().sort_index()
    print(f"\n  {label}:")
    for n_records, n_students in counts.items():
        print(f"    {n_records} record(s): {n_students:,} students")

File                 Total Rows  Unique SSIDs  Duplicates   Dupe %
Initial 2022             71,083        62,946       8,137    11.4%
Initial 2023             71,637        65,264       6,373     8.9%
Initial 2024             76,539        70,231       6,308     8.2%
Summative 2022          332,779       302,476      30,303     9.1%
Summative 2023          319,636       296,681      22,955     7.2%
Summative 2024          313,603       291,920      21,683     6.9%

Duplicate breakdown (number of records per SSID):

  Initial 2022:
    1 record(s): 55,603 students
    2 record(s): 6,610 students
    3 record(s): 678 students
    4 record(s): 49 students
    5 record(s): 6 students

  Initial 2023:
    1 record(s): 59,363 students
    2 record(s): 5,471 students
    3 record(s): 390 students
    4 record(s): 38 students
    5 record(s): 2 students

  Initial 2024:
    1 record(s): 64,435 students
    2 record(s): 5,338 students
    3 record(s): 413 students
    4 record(s): 38 students
 

In [3]:
# --- Clean Initial files: keep first (earliest TestStartDate) per SSID ---
cleaned_dfs = {}

for label in ["Initial 2022", "Initial 2023", "Initial 2024"]:
    df = dfs[label].copy()
    df_clean = (
        df.sort_values("TestStartDate", ascending=True)
          .drop_duplicates(subset="SSID", keep="first")
          .reset_index(drop=True)
    )
    cleaned_dfs[label] = df_clean
    print(f"{label}: {len(df):,} → {len(df_clean):,} rows "
          f"(removed {len(df) - len(df_clean):,} duplicates)")

# --- Clean Summative files: keep highest OverallScaleScore per SSID ---
for label in ["Summative 2022", "Summative 2023", "Summative 2024"]:
    df = dfs[label].copy()
    df_clean = (
        df.sort_values("OverallScaleScore", ascending=False)
          .drop_duplicates(subset="SSID", keep="first")
          .reset_index(drop=True)
    )
    cleaned_dfs[label] = df_clean
    print(f"{label}: {len(df):,} → {len(df_clean):,} rows "
          f"(removed {len(df) - len(df_clean):,} duplicates)")

# --- Save cleaned files ---
output_dir = "../1095 Data/ELPAC/Cleaned"
os.makedirs(output_dir, exist_ok=True)

file_name_map = {
    "Initial 2022":   "InitialElpac_22_clean.csv",
    "Initial 2023":   "InitialElpac_23_clean.csv",
    "Initial 2024":   "InitialElpac_24_clean.csv",
    "Summative 2022": "SummativeElpac_22_clean.csv",
    "Summative 2023": "SummativeElpac_23_clean.csv",
    "Summative 2024": "SummativeElpac_24_clean.csv",
}

for label, df_clean in cleaned_dfs.items():
    out_path = os.path.join(output_dir, file_name_map[label])
    df_clean.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")

Initial 2022: 71,083 → 62,946 rows (removed 8,137 duplicates)
Initial 2023: 71,637 → 65,264 rows (removed 6,373 duplicates)
Initial 2024: 76,539 → 70,231 rows (removed 6,308 duplicates)
Summative 2022: 332,779 → 302,476 rows (removed 30,303 duplicates)
Summative 2023: 319,636 → 296,681 rows (removed 22,955 duplicates)
Summative 2024: 313,603 → 291,920 rows (removed 21,683 duplicates)
Saved: ../1095 Data/ELPAC/Cleaned/InitialElpac_22_clean.csv
Saved: ../1095 Data/ELPAC/Cleaned/InitialElpac_23_clean.csv
Saved: ../1095 Data/ELPAC/Cleaned/InitialElpac_24_clean.csv
Saved: ../1095 Data/ELPAC/Cleaned/SummativeElpac_22_clean.csv
Saved: ../1095 Data/ELPAC/Cleaned/SummativeElpac_23_clean.csv
Saved: ../1095 Data/ELPAC/Cleaned/SummativeElpac_24_clean.csv


# looking at duplicate data

In [4]:
# --- Load original (uncleaned) Initial 2022 ---
initial_22_raw = pd.read_csv(os.path.join("../1095 Data/ELPAC/", "InitialElpac_22.csv"), low_memory=False)
initial_22_raw = initial_22_raw.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
initial_22_raw["TestStartDate"] = pd.to_datetime(initial_22_raw["TestStartDate"], errors="coerce")
initial_22_raw["OverallScaleScore"] = pd.to_numeric(initial_22_raw["OverallScaleScore"], errors="coerce")

# --- Find SSIDs with more than one record ---
dupe_ssids = (
    initial_22_raw.groupby("SSID")
    .filter(lambda x: len(x) > 1)["SSID"]
    .unique()
)

print(f"Students with duplicate Initial 2022 records: {len(dupe_ssids):,}")

# --- Show all records for duplicate students ---
dupes_df = (
    initial_22_raw[initial_22_raw["SSID"].isin(dupe_ssids)]
    .sort_values(["SSID", "TestStartDate"])
)

print(dupes_df[["SSID", "CALPADSSchoolCode", "OverallScaleScore", "OverallPL",
                "TestStartDate", "TestCompletedDate"]].to_string(index=False))

Students with duplicate Initial 2022 records: 7,343
      SSID  CALPADSSchoolCode  OverallScaleScore                                OverallPL TestStartDate TestCompletedDate
1002765014     19647336019764              151.0 Novice English Learner                      2021-09-10        2021-09-10
1002765014     19647336019764              151.0 Novice English Learner                      2021-09-10        2021-09-10
1002978573      1612596096523              226.0 Novice English Learner                      2021-08-04        2021-08-04
1002978573      1612596096523              226.0 Novice English Learner                      2021-08-04        2021-08-04
1002978573      1612596096523              226.0 Novice English Learner                      2021-08-04        2021-08-04
1004250194      7617546004048              400.0 Intermediate English Learner                2021-09-07        2021-09-07
1004250194      7617546004048              400.0 Intermediate English Learner                2

In [6]:
# --- Load original (uncleaned) Initial 2022 ---
initial_22_raw = pd.read_csv(os.path.join("../1095 Data/ELPAC/", "InitialElpac_22.csv"), low_memory=False)
initial_22_raw = initial_22_raw.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
initial_22_raw["TestStartDate"] = pd.to_datetime(initial_22_raw["TestStartDate"], errors="coerce")
initial_22_raw["OverallScaleScore"] = pd.to_numeric(initial_22_raw["OverallScaleScore"], errors="coerce")

# --- Find SSIDs with more than one record ---
dupe_ssids = (
    initial_22_raw.groupby("SSID")
    .filter(lambda x: len(x) > 1)["SSID"]
    .unique()
)

print(f"Students with duplicate Initial 2022 records: {len(dupe_ssids):,}\n")

# --- Display grouped with separator ---
cols = ["SSID", "CALPADSSchoolCode", "OverallScaleScore", "OverallPL",
        "TestStartDate", "TestCompletedDate"]

separator = "-" * 95

for ssid in sorted(dupe_ssids):
    student_records = initial_22_raw[initial_22_raw["SSID"] == ssid][cols].sort_values("TestStartDate")
    print(separator)
    print(student_records.to_string(index=False))

print(separator)
print(f"\nTotal duplicate students: {len(dupe_ssids):,}")

Students with duplicate Initial 2022 records: 7,343

-----------------------------------------------------------------------------------------------
      SSID  CALPADSSchoolCode  OverallScaleScore                                OverallPL TestStartDate TestCompletedDate
1002765014     19647336019764              151.0 Novice English Learner                      2021-09-10        2021-09-10
1002765014     19647336019764              151.0 Novice English Learner                      2021-09-10        2021-09-10
-----------------------------------------------------------------------------------------------
      SSID  CALPADSSchoolCode  OverallScaleScore                                OverallPL TestStartDate TestCompletedDate
1002978573      1612596096523              226.0 Novice English Learner                      2021-08-04        2021-08-04
1002978573      1612596096523              226.0 Novice English Learner                      2021-08-04        2021-08-04
1002978573      1612596